In [ ]:
# 交点パラメータ
import Pkg

c = ["Metal"]
Pkg.add(c)

using Metal 

# カーネル関数を使った交点を示すパラメータ(s, t)の計算
function kernel_s_t(s::MtlDeviceArray{Float32, 3}, t::MtlDeviceArray{Float32, 3},
                    ab_x::MtlDeviceArray{Float32, 3}, ab_y::MtlDeviceArray{Float32, 3},
                    ac_x::MtlDeviceArray{Float32, 3}, ac_y::MtlDeviceArray{Float32, 3},
                    ca_x::MtlDeviceArray{Float32, 3}, ca_y::MtlDeviceArray{Float32, 3},
                    cd_x::MtlDeviceArray{Float32, 3}, cd_y::MtlDeviceArray{Float32, 3})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(s, 1) || j > size(s, 2) || k > size(s, 3)
        return nothing
    end

    s[i, j, k] = (ac_x[i, j, k] * cd_y[i, j, k] - ac_y[i, j, k] * cd_x[i, j, k]) /
                (ab_x[i, j, k] * cd_y[i, j, k] - ab_y[i, j, k] * cd_x[i, j, k])
    t[i, j, k] = (ca_x[i, j, k] * ab_y[i, j, k] - ca_y[i, j, k] * ab_x[i, j, k]) /
                (cd_x[i, j, k] * ab_y[i, j, k] - cd_y[i, j, k] * ab_x[i, j, k])

    return nothing

end

function main()
    num_points = 31209
    div_n = 850
    num_edges = 176

    # num_points = 312
    # div_n = 8
    # num_edges = 17

    # デバイスメモリの確保
    Mtl_ab_x = MtlArray{Float32}(rand(Float32, num_points, div_n, num_edges))
    Mtl_ab_y = MtlArray{Float32}(rand(Float32, num_points, div_n, num_edges))
    Mtl_ac_x = MtlArray{Float32}(rand(Float32, num_points, div_n, num_edges))
    Mtl_ac_y = MtlArray{Float32}(rand(Float32, num_points, div_n, num_edges))
    Mtl_cd_x = MtlArray{Float32}(rand(Float32, num_points, div_n, num_edges))
    Mtl_cd_y = MtlArray{Float32}(rand(Float32, num_points, div_n, num_edges))

    Mtl_ca_x = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    Mtl_ca_y = similar(Mtl_ca_x)
    Mtl_ca_x .= -Mtl_ab_x
    Mtl_ca_y .= -Mtl_ab_y

   threads_per_group = (8, 8, 4)  #  <= 512
    num_groups = (ceil(Int, num_points / threads_per_group[1]), 
                ceil(Int, div_n / threads_per_group[2]), 
                ceil(Int, num_edges / threads_per_group[3]))
 

Mtl_s = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    Mtl_t = similar(Mtl_s)
    @metal threads = threads_per_group groups = num_groups kernel_s_t(Mtl_s, Mtl_t,
                                                                    Mtl_ab_x, Mtl_ab_y,
                                                                    Mtl_ac_x, Mtl_ac_y,
                                                                    Mtl_ca_x, Mtl_ca_y,
                                                                    Mtl_cd_x, Mtl_cd_y)
                                                                    
s = Array(Mtl_s)

    # デバイスメモリの解放
    Mtl_s = nothing
    Mtl_t = nothing
    Mtl_ab_x = nothing
    Mtl_ab_y = nothing
    Mtl_ac_x = nothing
    Mtl_ac_y = nothing
    Mtl_ca_x = nothing
    Mtl_ca_y = nothing
    Mtl_cd_x = nothing
    Mtl_cd_y = nothing
# 結果の表示
@show s

end

main()

In [2]:
# gemini
# 点(xx, yy)を始点とする半直線abが点(x0, y0)および点(x1, y1)を端点とする線分cdと交わるかを判断し，交わる場合は交点座標を計算するJuliaのコードを生成して

function intersect_ray_segment(xx, yy, a, b, x0, y0, x1, y1)
  """
  点(xx, yy)を始点とする半直線abが点(x0, y0)および点(x1, y1)を端点とする線分cdと交わるかを判断し，
  交わる場合は交点座標を計算します。

  Args:
    xx: 半直線abの始点のx座標
    yy: 半直線abの始点のy座標
    a: 半直線abの方向ベクトルのx成分 (点bのx座標 - xx)
    b: 半直線abの方向ベクトルのy成分 (点bのy座標 - yy)
    x0: 線分cdの端点cのx座標
    y0: 線分cdの端点cのy座標
    x1: 線分cdの端点dのx座標
    y1: 線分cdの端点dのy座標

  Returns:
    交差しない場合は nothing を返します。
    交差する場合は，交点の座標 (x, y) を返します。
  """

  # 線分cdの方向ベクトル
  c = x1 - x0
  d = y1 - y0

  # 行列式
  det = a * d - b * c

  # 平行な場合は交差しない
  if det == 0
    return nothing
  end

  # パラメータtとuを計算
  t = ((x0 - xx) * d - (y0 - yy) * c) / det
  u = ((x0 - xx) * b - (y0 - yy) * a) / det

  # 半直線との交差条件 (t >= 0) および線分との交差条件 (0 <= u <= 1) を満たす場合
  if t >= 0 && 0 <= u <= 1
    # 交点の座標を計算
    intersection_x = xx + t * a
    intersection_y = yy + t * b
    return (intersection_x, intersection_y)
  else
    return nothing
  end
end

xx = 0.0
yy = 0.0
a = 1.0
b = 1.0
x0 = 1.0
y0 = 0.0
x1 = 0.0
y1 = 1.0

intersection = intersect_ray_segment(xx, yy, a, b, x0, y0, x1, y1)

if intersection === nothing
  println("半直線と線分は交差しません。")
else
  println("半直線と線分の交点: $(intersection)")
end



半直線と線分の交点: (0.5, 0.5)
